In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()


DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "testdb")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)

In [2]:
# Windowed project coverage: 2018-01-01 to 2020-10-31 (selected projects only)
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]
project_keys_sql = ", ".join([f"'{k}'" for k in project_keys])

sql = f"""
WITH created AS (
  SELECT
    i.Project_ID,
    COUNT(*) AS issues_created_2018_2020,
    SUM(i.Creation_Date >= '2018-01-01 00:00:00' AND i.Creation_Date < '2019-01-01 00:00:00') AS issues_created_2018,
    SUM(i.Creation_Date >= '2019-01-01 00:00:00' AND i.Creation_Date < '2020-01-01 00:00:00') AS issues_created_2019,
    SUM(i.Creation_Date >= '2020-01-01 00:00:00' AND i.Creation_Date < '2021-01-01 00:00:00') AS issues_created_2020,
    MIN(i.Creation_Date) AS first_created_in_window,
    MAX(i.Creation_Date) AS last_created_in_window
  FROM Issue i
  WHERE i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
  GROUP BY i.Project_ID
),
resolved AS (
  SELECT
    i.Project_ID,
    COUNT(*) AS issues_resolved_2018_2020,
    SUM(i.Resolution_Date >= '2018-01-01 00:00:00' AND i.Resolution_Date < '2019-01-01 00:00:00') AS issues_resolved_2018,
    SUM(i.Resolution_Date >= '2019-01-01 00:00:00' AND i.Resolution_Date < '2020-01-01 00:00:00') AS issues_resolved_2019,
    SUM(i.Resolution_Date >= '2020-01-01 00:00:00' AND i.Resolution_Date < '2021-01-01 00:00:00') AS issues_resolved_2020,
    MIN(i.Resolution_Date) AS first_resolved_in_window,
    MAX(i.Resolution_Date) AS last_resolved_in_window
  FROM Issue i
  WHERE i.Resolution_Date IS NOT NULL
    AND i.Resolution_Date BETWEEN '{window_start}' AND '{window_end}'
  GROUP BY i.Project_ID
)
SELECT
  p.Project_Key,
  p.Name,
  COALESCE(c.issues_created_2018_2020, 0) AS issues_created_2018_2020,
  COALESCE(r.issues_resolved_2018_2020, 0) AS issues_resolved_2018_2020,
  COALESCE(c.issues_created_2018, 0) AS issues_created_2018,
  COALESCE(c.issues_created_2019, 0) AS issues_created_2019,
  COALESCE(c.issues_created_2020, 0) AS issues_created_2020,
  COALESCE(r.issues_resolved_2018, 0) AS issues_resolved_2018,
  COALESCE(r.issues_resolved_2019, 0) AS issues_resolved_2019,
  COALESCE(r.issues_resolved_2020, 0) AS issues_resolved_2020,
  COALESCE(r.issues_resolved_2018_2020, 0) / NULLIF(COALESCE(c.issues_created_2018_2020, 0), 0) AS resolved_rate_2018_2020,
  c.first_created_in_window,
  c.last_created_in_window,
  r.first_resolved_in_window,
  r.last_resolved_in_window
FROM Project p
LEFT JOIN created c ON c.Project_ID = p.ID
LEFT JOIN resolved r ON r.Project_ID = p.ID
WHERE p.Project_Key IN ({project_keys_sql})
ORDER BY issues_created_2018_2020 DESC, p.Project_Key;
"""

df = q(sql)
out = Path("../reports/first_look/project_coverage_2018_2020.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
out
# ...existing code...

PosixPath('../reports/first_look/project_coverage_2018_2020.csv')

In [3]:
# Simple profiling for selected projects (issues created in window)
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]
project_keys_sql = ", ".join([f"'{k}'" for k in project_keys])

sql = f"""
WITH base_issues AS (
  SELECT
    i.ID AS Issue_ID,
    i.Project_ID,
    i.Resolution_Date,
    i.Resolution_Time_Minutes
  FROM Issue i
  WHERE i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
),
issue_counts AS (
  SELECT
    Project_ID,
    COUNT(*) AS n_issues_2018_2020,
    SUM(Resolution_Date IS NOT NULL) AS n_resolved_2018_2020,
    SUM(Resolution_Time_Minutes IS NOT NULL) AS n_with_resolution_time_minutes
  FROM base_issues
  GROUP BY Project_ID
),
changelog_counts AS (
  SELECT
    bi.Project_ID,
    bi.Issue_ID,
    COUNT(*) AS n_changelog_entries,
    SUM(LOWER(cl.Field) = 'status') AS n_status_changes,
    SUM(LOWER(cl.Field) IN ('assignee', 'reporter')) AS n_people_changes
  FROM base_issues bi
  LEFT JOIN Change_Log cl ON cl.Issue_ID = bi.Issue_ID
  GROUP BY bi.Project_ID, bi.Issue_ID
),
changelog_rollup AS (
  SELECT
    Project_ID,
    SUM(n_changelog_entries > 0) AS n_with_any_changelog,
    SUM(n_status_changes > 0) AS n_with_status_changes,
    SUM(n_people_changes > 0) AS n_with_people_changes,
    AVG(n_changelog_entries) AS avg_changelog_entries_per_issue
  FROM changelog_counts
  GROUP BY Project_ID
),
comments_counts AS (
  SELECT
    bi.Project_ID,
    bi.Issue_ID,
    COUNT(c.ID) AS n_comments
  FROM base_issues bi
  LEFT JOIN Comment c ON c.Issue_ID = bi.Issue_ID
  GROUP BY bi.Project_ID, bi.Issue_ID
),
comments_rollup AS (
  SELECT
    Project_ID,
    AVG(n_comments) AS avg_comments_per_issue
  FROM comments_counts
  GROUP BY Project_ID
)
SELECT
  p.Project_Key,
  ic.n_issues_2018_2020,
  ic.n_resolved_2018_2020,
  ic.n_resolved_2018_2020 / NULLIF(ic.n_issues_2018_2020, 0) AS resolved_rate,
  ic.n_with_resolution_time_minutes,
  cr.n_with_any_changelog,
  cr.n_with_status_changes,
  cr.n_with_people_changes,
  cr.avg_changelog_entries_per_issue AS median_changelog_entries_per_issue,
  cor.avg_comments_per_issue AS median_comments_per_issue
FROM Project p
JOIN issue_counts ic ON ic.Project_ID = p.ID
LEFT JOIN changelog_rollup cr ON cr.Project_ID = p.ID
LEFT JOIN comments_rollup cor ON cor.Project_ID = p.ID
WHERE p.Project_Key IN ({project_keys_sql})
ORDER BY ic.n_issues_2018_2020 DESC, p.Project_Key;
"""

df_profile = q(sql)
out = Path("../reports/first_look/issues_profile_2018_2020.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df_profile.to_csv(out, index=False)
out


PosixPath('../reports/first_look/issues_profile_2018_2020.csv')

In [4]:
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]

for project_key in project_keys:
    sql = f"""
    WITH base_issues AS (
      SELECT
        p.Project_Key AS Project_Key,
        i.ID AS Issue_ID,
        i.Issue_Key AS Issue_Key,
        i.Type AS Issue_Type,
        i.Creation_Date AS Created,
        i.Resolution_Date AS Resolution,
        i.Resolution_Time_Minutes AS Resolution_Time_Minutes,
        i.In_Progress_Minutes AS In_Progress_Minutes
      FROM Issue i
      JOIN Project p ON p.ID = i.Project_ID
      WHERE p.Project_Key = '{project_key}'
        AND i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
        AND i.Resolution_Date IS NOT NULL
    ),
    status_events AS (
      SELECT
        bi.Project_Key,
        bi.Issue_ID,
        cl.ID AS Change_Log_ID,
        cl.Creation_Date AS Change_Date,
        LOWER(TRIM(COALESCE(NULLIF(cl.From_String, ''), cl.From_Value))) AS From_Status,
        LOWER(TRIM(COALESCE(NULLIF(cl.To_String, ''), cl.To_Value))) AS To_Status
      FROM base_issues bi
      JOIN Change_Log cl
        ON cl.Issue_ID = bi.Issue_ID
      WHERE LOWER(TRIM(cl.Field)) = 'status'
    ),
    first_status_event AS (
      SELECT
        x.Project_Key,
        x.Issue_ID,
        x.Change_Date AS First_Status_Transition,
        x.To_Status AS First_Status_To
      FROM (
        SELECT
          se.*,
          ROW_NUMBER() OVER (
            PARTITION BY se.Project_Key, se.Issue_ID
            ORDER BY se.Change_Date, se.Change_Log_ID
          ) AS rn
        FROM status_events se
      ) x
      WHERE x.rn = 1
    ),
    status_rollup AS (
      SELECT
        se.Project_Key,
        se.Issue_ID,
        COUNT(*) AS Num_Status_Changes,
        MIN(CASE
              WHEN se.To_Status = 'in progress'
                OR se.To_Status = 'inprogress'
                OR se.To_Status LIKE '%%in progress%%'
              THEN se.Change_Date
            END) AS First_In_Progress_Timestamp,
        MIN(CASE
              WHEN se.To_Status IN ('done', 'closed', 'resolved', 'verified')
              THEN se.Change_Date
            END) AS First_Done_Timestamp,
        MAX(CASE
              WHEN se.To_Status IN ('done', 'closed', 'resolved', 'verified')
              THEN se.Change_Date
            END) AS Final_Done_Timestamp,
        MAX(CASE
              WHEN se.From_Status IN ('done', 'closed', 'resolved', 'verified')
               AND se.To_Status NOT IN ('done', 'closed', 'resolved', 'verified')
              THEN 1 ELSE 0
            END) AS Was_Reopened_Int
      FROM status_events se
      GROUP BY se.Project_Key, se.Issue_ID
    )
    SELECT
      bi.Project_Key,
      bi.Issue_ID,
      bi.Issue_Key,
      bi.Issue_Type,
      bi.Created,
      bi.Resolution,
      (sr.Issue_ID IS NOT NULL) AS Has_Status_History,
      COALESCE(sr.Num_Status_Changes, 0) AS Num_Status_Changes,
      fse.First_Status_Transition,
      fse.First_Status_To,
      sr.First_In_Progress_Timestamp,
      sr.First_Done_Timestamp,
      sr.Final_Done_Timestamp,
      COALESCE(sr.Was_Reopened_Int, 0) AS Was_Reopened,
      (bi.Resolution_Time_Minutes IS NOT NULL) AS Has_Resolution_Time_Minutes,
      bi.Resolution_Time_Minutes,
      bi.In_Progress_Minutes
    FROM base_issues bi
    LEFT JOIN status_rollup sr
      ON sr.Project_Key = bi.Project_Key
     AND sr.Issue_ID = bi.Issue_ID
    LEFT JOIN first_status_event fse
      ON fse.Project_Key = bi.Project_Key
     AND fse.Issue_ID = bi.Issue_ID
    ORDER BY bi.Created, bi.Issue_ID;
    """

    df_rel = q(sql)
    out = Path(f"../reports/first_look/status_history_reliability_{project_key}_2018_2020.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    df_rel.to_csv(out, index=False)

out

PosixPath('../reports/first_look/status_history_reliability_JRACLOUD_2018_2020.csv')

In [5]:
# Cache resolved issue-level base table for selected projects (reusable)
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]
project_keys_sql = ", ".join([f"'{k}'" for k in project_keys])

issues_df = q(f"""
SELECT
  p.Project_Key,
  i.ID AS Issue_ID,
  i.Issue_Key,
  i.Type AS Issue_Type,
  i.Creation_Date AS Created,
  i.Resolution_Date AS Resolution,
  i.Resolution_Time_Minutes
FROM Issue i
JOIN Project p ON p.ID = i.Project_ID
WHERE p.Project_Key IN ({project_keys_sql})
  AND i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
  AND i.Resolution_Date IS NOT NULL;
""")

out = Path("../data/interim/issues_SERVER_MDL_JRACLOUD_2018_2020.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
issues_df.to_parquet(out, index=False)
out

PosixPath('../data/interim/issues_SERVER_MDL_JRACLOUD_2018_2020.parquet')

In [6]:
# Get the top 20 used statuses in each project
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]

for project_key in project_keys:
    sql = f"""
    SELECT
      cl.To_String,
      COUNT(*) AS n
    FROM Change_Log cl
    JOIN Issue i ON i.ID = cl.Issue_ID
    JOIN Project p ON p.ID = i.Project_ID
    WHERE p.Project_Key = '{project_key}'
      AND i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
      AND LOWER(cl.Field) = 'status'
    GROUP BY cl.To_String
    ORDER BY n DESC
    LIMIT 20;
    """
    status_vals = q(sql)

    out = Path(f"../reports/first_look/{project_key}_status_to_values_top20_2018_2020.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    status_vals.to_csv(out, index=False)

out


PosixPath('../reports/first_look/JRACLOUD_status_to_values_top20_2018_2020.csv')

In [7]:
# Get the top 10 used priorities in each project
window_start = "2018-01-01 00:00:00"
window_end = "2020-10-31 23:59:59"
project_keys = ["SERVER", "MDL", "JRACLOUD"]

for project_key in project_keys:
    sql = f"""
    SELECT
      cl.To_String,
      COUNT(*) AS n
    FROM Change_Log cl
    JOIN Issue i ON i.ID = cl.Issue_ID
    JOIN Project p ON p.ID = i.Project_ID
    WHERE p.Project_Key = '{project_key}'
      AND i.Creation_Date BETWEEN '{window_start}' AND '{window_end}'
      AND LOWER(cl.Field) = 'priority'
    GROUP BY cl.To_String
    ORDER BY n DESC
    LIMIT 10;
    """
    status_vals = q(sql)

    out = Path(f"../reports/first_look/{project_key}_priority_to_values_top10_2018_2020.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    status_vals.to_csv(out, index=False)

out

PosixPath('../reports/first_look/JRACLOUD_priority_to_values_top10_2018_2020.csv')

In [8]:
# Get the coverage for the six candidate cycle times

# CT1 = Resolution_Time_Minutes
# CT2 = In_Progress_Minutes
# CT3 = Resolution - Created
# CT4 = Resolution - First_In_Progress_Timestamp
# CT5 = Final_Done_Timestamp - First_In_Progress_Timestamp
# CT6 = First_Done_Timestamp - First_In_Progress_Timestamp


import pandas as pd
import numpy as np
from pathlib import Path

project_keys = ["SERVER", "MDL", "JRACLOUD"]

rows = []

for project_key in project_keys:
    path = Path(f"../reports/first_look/status_history_reliability_{project_key}_2018_2020.csv")
    df = pd.read_csv(
        path,
        parse_dates=[
            "Created",
            "Resolution",
            "First_Status_Transition",
            "First_In_Progress_Timestamp",
            "First_Done_Timestamp",
            "Final_Done_Timestamp",
        ],
    )

    n = len(df)

    # Derived durations in minutes
    ct3_minutes = (df["Resolution"] - df["Created"]).dt.total_seconds() / 60
    ct4_minutes = (df["Resolution"] - df["First_In_Progress_Timestamp"]).dt.total_seconds() / 60
    ct5_minutes = (df["Final_Done_Timestamp"] - df["First_In_Progress_Timestamp"]).dt.total_seconds() / 60
    ct6_minutes = (df["First_Done_Timestamp"] - df["First_In_Progress_Timestamp"]).dt.total_seconds() / 60

    rows.append({
        "Project": project_key,
        "Issues": n,

        # Basic field availability
        "Has_Status_History": int(df["Has_Status_History"].fillna(0).astype(bool).sum()),
        "Resolution_nonnull": int(df["Resolution"].notna().sum()),
        "First_In_Progress_nonnull": int(df["First_In_Progress_Timestamp"].notna().sum()),
        "First_Done_nonnull": int(df["First_Done_Timestamp"].notna().sum()),
        "Final_Done_nonnull": int(df["Final_Done_Timestamp"].notna().sum()),

        # CT1
        "CT1_count_nonnull": int(df["Resolution_Time_Minutes"].notna().sum()),
        "CT1_count_gt0": int((df["Resolution_Time_Minutes"].fillna(0) > 0).sum()),

        # CT2
        "CT2_count_nonnull": int(df["In_Progress_Minutes"].notna().sum()),
        "CT2_count_gt0": int((df["In_Progress_Minutes"].fillna(0) > 0).sum()),

        # CT3
        "CT3_count_nonnull": int(ct3_minutes.notna().sum()),
        "CT3_count_nonneg": int((ct3_minutes >= 0).sum()),

        # CT4
        "CT4_count_nonnull": int(ct4_minutes.notna().sum()),
        "CT4_count_nonneg": int((ct4_minutes >= 0).sum()),

        # CT5
        "CT5_count_nonnull": int(ct5_minutes.notna().sum()),
        "CT5_count_nonneg": int((ct5_minutes >= 0).sum()),

        # CT6
        "CT6_count_nonnull": int(ct6_minutes.notna().sum()),
        "CT6_count_nonneg": int((ct6_minutes >= 0).sum()),

        # Optional consistency checks
        "CT1_vs_CT3_mismatch": int(
            (
                df["Resolution_Time_Minutes"].notna()
                & ct3_minutes.notna()
                & ~np.isclose(
                    df["Resolution_Time_Minutes"].astype(float),
                    ct3_minutes.astype(float),
                    equal_nan=False
                )
            ).sum()
        ),
        "CT2_vs_CT4_mismatch": int(
            (
                df["In_Progress_Minutes"].notna()
                & ct4_minutes.notna()
                & ~np.isclose(
                    df["In_Progress_Minutes"].astype(float),
                    ct4_minutes.astype(float),
                    equal_nan=False
                )
            ).sum()
        ),
    })

coverage_df = pd.DataFrame(rows)
coverage_df
coverage_df.to_csv("coverage_counts.csv", index=False)
